# Étape 4 - Feature engineering sur le jeu de prédiction (online)

Le jeu de prédiction est **dans le futur** : il n'existe pas encore, et surtout
il n'a pas d'historique pour calculer `lag_1W` (le CA d'il y a une semaine).

## Deux façons de calculer le lag

| Méthode | Principe | Inconvénient |
|---|---|---|
| coûteuse en RAM | concaténer **tout** `train` + `future` puis `shift` | occupe beaucoup de mémoire pour peu d'info utile |
| **recommandée** (`lag_online`) | charger seulement la **semaine passée** (`past`), concaténer avec `future`, puis `shift` | aucun — on n'a besoin que d'une semaine |

## Les 4 briques

| Fonction | Rôle |
|---|---|
| `span_future` | génère les dates futures à prédire (1 semaine, au pas horaire) |
| `dummy_day` | jour de la semaine en 6 binaires |
| `hour_cos_sin` | heure en cos/sin |
| `lag_online` | `lag_1W` calculé à partir de `past` uniquement |

`span_future` est autonome ; les 3 autres sont enchaînées par **`features_online`**.

## 1. Importer les librairies

In [ ]:
import sys
sys.path.append('..')
import yaml
import logging
import logging.config
import numpy as np
import pandas as pd
pd.set_option('display.min_rows', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 500)
pd.set_option('max_colwidth', 400)

from foodcast.domain.transform import etl
from foodcast.domain.feature_engineering import features_offline, features_online
from foodcast.domain.forecast import span_future, cross_validate, plotly_predictions
from foodcast.domain.multi_model import MultiModel
from sklearn.ensemble import RandomForestRegressor
import foodcast.settings as settings
import plotly.graph_objects as go

with open(settings.LOGGING_CONFIGURATION_FILE, 'r') as f:
    logging.config.dictConfig(yaml.safe_load(f.read()))

%load_ext autoreload
%autoreload 2

## 2. Construire `past` : la semaine juste avant la prédiction

On prédit la semaine qui suit la semaine 200 → `past` = semaine 200, nettoyée avec `etl`.

In [ ]:
past = etl(settings.DATA_DIR, 200, 200)
past.tail()

## 3. Regarder le code de `span_future`

In [ ]:
span_future??

## 4. Générer les dates futures

On part de la dernière date connue dans `past`. `span_future` démarre au minuit suivant
et couvre 1 semaine au pas d'1 heure.

In [ ]:
future = span_future(past['order_date'].max())
future.head()

In [ ]:
future.shape  # 7 jours x 24 heures = 168 lignes

## 5. Regarder le code de `features_online`

In [ ]:
features_online??

## 6. Construire le jeu de prédiction complet

`features_online(df, past)` : `df` = les dates futures, `past` = la semaine précédente
pour le lag.

In [ ]:
future = features_online(future, past)
future.head(20)

## 7. Vérifier une ligne à la main

On regarde le 5 novembre 2018 à 18h : `lag_1W` doit valoir le `cash_in`
du 29 octobre 2018 à 18h (présent dans `past`).

In [ ]:
future[future['order_date'] == '2018-11-05 18:00:00']

In [ ]:
resample_past = past.set_index('order_date')
resample_past.loc['2018-10-29 18:00:00', 'cash_in']  # doit correspondre au lag_1W ci-dessus

## 8. Mettre la date dans l'index

Comme pour `x_train`, on garde la date en index (et non en colonne).

In [ ]:
future = future.set_index('order_date')
future.head()

`future` a maintenant exactement les mêmes colonnes que `x_train`.

➡️ Étape suivante : `05_prevision_visualisation.ipynb`